# Export Model for Live Trading

Train LogisticRegression on all legacy data, save model + scaler + feature columns to `models/` for use by polybot.

In [1]:
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

## 1. Load legacy training data

In [2]:
rows = []
with open(Path("../data/legacy_features.jsonl")) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

NON_FEAT = {
    "candle_id",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
feat_cols = sorted([c for c in df.columns if c not in NON_FEAT])
df[feat_cols] = df[feat_cols].fillna(0.0)

print(f"Training data: {len(df):,} rows, {df['candle_id'].nunique()} candles, {len(feat_cols)} features")
print(f"UP rate: {df['target'].mean() * 100:.1f}%")

Training data: 94,336 rows, 869 candles, 56 features
UP rate: 48.7%


## 2. Train model

In [3]:
scaler = StandardScaler()
X_train = scaler.fit_transform(df[feat_cols].values)
y_train = df["target"].values

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
print(f"Train accuracy: {train_acc * 100:.1f}%")
print(f"Coefficients: {model.coef_.shape}")

Train accuracy: 75.9%
Coefficients: (1, 56)


## 3. Export to disk

In [4]:
joblib.dump(model, MODELS_DIR / "logistic_v1.joblib")
joblib.dump(scaler, MODELS_DIR / "scaler_v1.joblib")
joblib.dump(feat_cols, MODELS_DIR / "feature_cols_v1.joblib")

print(f"Saved to {MODELS_DIR}:")
for f in sorted(MODELS_DIR.glob("*_v1.*")):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

Saved to ../models:
  feature_cols_v1.joblib (1,132 bytes)
  logistic_v1.joblib (1,311 bytes)
  scaler_v1.joblib (1,927 bytes)


## 4. Verify load

In [5]:
# Verify round-trip
m2 = joblib.load(MODELS_DIR / "logistic_v1.joblib")
s2 = joblib.load(MODELS_DIR / "scaler_v1.joblib")
fc2 = joblib.load(MODELS_DIR / "feature_cols_v1.joblib")

# Predict on first row
sample = df[fc2].iloc[:1].fillna(0.0).values
prob_orig = model.predict_proba(scaler.transform(sample))[0, 1]
prob_loaded = m2.predict_proba(s2.transform(sample))[0, 1]

print(f"Original prob:  {prob_orig:.6f}")
print(f"Loaded prob:    {prob_loaded:.6f}")
print(f"Match: {prob_orig == prob_loaded}")
print(f"\nFeature columns ({len(fc2)}): {fc2[:5]} ... {fc2[-5:]}")

Original prob:  0.984302
Loaded prob:    0.984302
Match: True

Feature columns (56): ['adx', 'bollinger_pct_b', 'btc_acceleration', 'btc_direction_consistency', 'btc_drawdown_from_peak'] ... ['up_implied_probability', 'up_risk_reward', 'up_spread_level', 'up_token_velocity', 'weighted_mid_price']
